# Vendas de Derivados — Perfil Exploratório

Análise consolidada das séries de vendas (mensal, segmento, diesel/tipo, GLP/vasilhame, biodiesel, municipal).

**Fonte:** ANP — Dados Abertos — Vendas de derivados de petróleo e biocombustíveis  
**Trusted:** `data/trusted/vendas-derivados/`

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

REPO = Path.cwd().parents[2]
TRUSTED = REPO / 'data' / 'trusted' / 'vendas-derivados'

mensal = pd.read_parquet(TRUSTED / 'vendas_mensal.parquet')
segmento = pd.read_parquet(TRUSTED / 'vendas_segmento.parquet')
diesel = pd.read_parquet(TRUSTED / 'vendas_diesel_tipo.parquet')
glp = pd.read_parquet(TRUSTED / 'vendas_glp_vasilhame.parquet')
biodiesel = pd.read_parquet(TRUSTED / 'vendas_biodiesel.parquet')
municipal = pd.read_parquet(TRUSTED / 'vendas_municipal.parquet')

print(f'Mensal:     {mensal.shape[0]:>10,} linhas  ({mensal.ano.min()}–{mensal.ano.max()})')
print(f'Segmento:   {segmento.shape[0]:>10,} linhas  ({segmento.ano.min()}–{segmento.ano.max()})')
print(f'Diesel tipo:{diesel.shape[0]:>10,} linhas  ({diesel.ano.min()}–{diesel.ano.max()})')
print(f'GLP vasil.: {glp.shape[0]:>10,} linhas  ({glp.ano.min()}–{glp.ano.max()})')
print(f'Biodiesel:  {biodiesel.shape[0]:>10,} linhas  ({biodiesel.ano.min()}–{biodiesel.ano.max()})')
print(f'Municipal:  {municipal.shape[0]:>10,} linhas  ({municipal.ano.min()}–{municipal.ano.max()})')

## 1. Evolução temporal — vendas mensais totais

In [ ]:
total_mes = mensal.groupby('data_referencia')['vendas_m3'].sum().reset_index()
total_mes['data_referencia'] = pd.to_datetime(total_mes['data_referencia'])

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(total_mes['data_referencia'], total_mes['vendas_m3'] / 1e6, linewidth=0.8)
ax.set_title('Vendas totais de derivados — série mensal (1990–2026)')
ax.set_ylabel('Milhões m³')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 2. Vendas por produto — top 5

In [ ]:
top5 = mensal.groupby('produto')['vendas_m3'].sum().nlargest(5).index.tolist()

fig, ax = plt.subplots(figsize=(14, 5))
for prod in top5:
    serie = mensal[mensal['produto'] == prod].groupby('data_referencia')['vendas_m3'].sum()
    serie.index = pd.to_datetime(serie.index)
    ax.plot(serie.index, serie.values / 1e6, label=prod, linewidth=0.8)

ax.set_title('Top 5 produtos — vendas mensais')
ax.set_ylabel('Milhões m³')
ax.legend(fontsize=8)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 3. Sazonalidade — vendas por mês do ano (últimos 5 anos)

In [ ]:
recente = mensal[mensal['ano'] >= 2020].copy()
saz = recente.groupby('mes')['vendas_m3'].mean() / 1e6

fig, ax = plt.subplots(figsize=(8, 4))
saz.plot(kind='bar', ax=ax, color='steelblue')
ax.set_title('Sazonalidade — média mensal 2020–2026')
ax.set_ylabel('Milhões m³')
ax.set_xlabel('Mês')
plt.tight_layout()
plt.show()

## 4. Diesel por tipo (S-10 vs S-500)

In [ ]:
diesel_agg = diesel.groupby(['data_referencia', 'produto'])['vendas_m3'].sum().unstack(fill_value=0)
diesel_agg.index = pd.to_datetime(diesel_agg.index)

fig, ax = plt.subplots(figsize=(14, 4))
for col in diesel_agg.columns:
    ax.plot(diesel_agg.index, diesel_agg[col] / 1e3, label=col, linewidth=0.8)
ax.set_title('Diesel por tipo — evolução mensal (mil m³)')
ax.set_ylabel('Mil m³')
ax.legend(fontsize=8)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 5. GLP — P13 vs outros vasilhames

In [ ]:
glp_agg = glp.groupby(['data_referencia', 'vasilhame'])['vendas_m3'].sum().unstack(fill_value=0)
glp_agg.index = pd.to_datetime(glp_agg.index)

fig, ax = plt.subplots(figsize=(14, 4))
for col in glp_agg.columns:
    ax.plot(glp_agg.index, glp_agg[col] / 1e3, label=col, linewidth=0.8)
ax.set_title('GLP por vasilhame — evolução mensal (mil m³)')
ax.set_ylabel('Mil m³')
ax.legend(fontsize=8)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Biodiesel — fluxo regional

In [ ]:
bio_flow = biodiesel.groupby(['regiao_origem', 'regiao_destino'])['vendas_m3'].sum().reset_index()
bio_flow = bio_flow.sort_values('vendas_m3', ascending=False)
print('Top 10 fluxos regionais (origem → destino):')
print(bio_flow.head(10).to_string(index=False))
print(f'\nTotal biodiesel: {biodiesel.vendas_m3.sum():,.0f} m³')

## 7. Municipal — top 20 municípios por volume (gasolina, último ano)

In [ ]:
ano_max = municipal['ano'].max()
gas_mun = municipal[(municipal['produto'] == 'GASOLINA C') & (municipal['ano'] == ano_max)]
top_mun = gas_mun.nlargest(20, 'vendas_m3')[['municipio', 'uf', 'vendas_m3']].copy()
top_mun['vendas_mil_m3'] = top_mun['vendas_m3'] / 1e3

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(top_mun['municipio'].str.strip() + ' - ' + top_mun['uf'],
        top_mun['vendas_mil_m3'], color='coral')
ax.set_xlabel('Mil m³')
ax.set_title(f'Top 20 municípios — Gasolina C ({ano_max})')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## 8. Vendas por UF — comparativo recente

In [ ]:
uf_2024 = mensal[mensal['ano'] == 2024].groupby('uf')['vendas_m3'].sum().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(12, 5))
uf_2024.plot(kind='bar', ax=ax, color='teal')
ax.set_title('Vendas totais por UF — 2024')
ax.set_ylabel('m³')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

## 9. Segmento — distribuição

In [ ]:
seg_total = segmento.groupby('segmento')['vendas_m3'].sum().sort_values(ascending=False)
print('Vendas por segmento (total 2012–2026):')
print((seg_total / 1e6).round(1).to_string())
print(f'\nTotal: {seg_total.sum()/1e6:.1f} milhões m³')